# 血糖预测 vs 真实对比

基于 OhioT1DM 真实患者数据：
- **输入**：运动、饮食（碳水）、胰岛素（bolus + basal）
- **输出**：模型预测的血糖变化曲线
- **对比**：与 CGM 真实血糖对比，标注事件时间点

In [ ]:
import importlib
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import xml.etree.ElementTree as ET
import sys, os

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

# 强制重新导入模块
import model.neural_ode_glucose
importlib.reload(model.neural_ode_glucose)
from model.neural_ode_glucose import NeuralODEGlucosePredictor

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'设备: {device}')

## 1. 解析 OhioT1DM XML 数据

In [ ]:
def parse_xml(xml_path):
    """解析 OhioT1DM XML 文件"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    patient_id = root.attrib['id']

    def parse_events(elem):
        rows = []
        if elem is None:
            return pd.DataFrame()
        for e in elem.findall('event'):
            row = dict(e.attrib)
            row['patient_id'] = patient_id
            rows.append(row)
        return pd.DataFrame(rows) if rows else pd.DataFrame()

    # glucose_level
    df_glucose = parse_events(root.find('glucose_level'))
    if not df_glucose.empty:
        df_glucose['timestamp'] = pd.to_datetime(df_glucose['ts'], format='%d-%m-%Y %H:%M:%S')
        df_glucose['glucose_level'] = df_glucose['value'].astype(float)
        df_glucose = df_glucose[['timestamp', 'glucose_level', 'patient_id']]

    # meal
    df_meal = parse_events(root.find('meal'))
    if not df_meal.empty:
        df_meal['timestamp'] = pd.to_datetime(df_meal['ts'], format='%d-%m-%Y %H:%M:%S')
        df_meal['carbs'] = df_meal['carbs'].astype(float)
        df_meal = df_meal[['timestamp', 'carbs', 'patient_id']]

    # exercise
    df_exercise = parse_events(root.find('exercise'))
    if not df_exercise.empty:
        df_exercise['timestamp'] = pd.to_datetime(df_exercise['ts'], format='%d-%m-%Y %H:%M:%S')
        df_exercise['intensity'] = df_exercise['intensity'].astype(float)
        df_exercise['duration'] = df_exercise['duration'].astype(float)
        df_exercise = df_exercise[['timestamp', 'intensity', 'duration', 'patient_id']]

    # heart_rate
    df_hr = parse_events(root.find('basis_heart_rate'))
    if not df_hr.empty:
        df_hr['timestamp'] = pd.to_datetime(df_hr['ts'], format='%d-%m-%Y %H:%M:%S')
        df_hr['heart_rate'] = df_hr['value'].astype(float)
        df_hr = df_hr[['timestamp', 'heart_rate', 'patient_id']]

    # sleep
    df_sleep = parse_events(root.find('sleep'))
    if not df_sleep.empty:
        df_sleep['sleep_start'] = pd.to_datetime(df_sleep['ts_begin'], format='%d-%m-%Y %H:%M:%S')
        df_sleep['sleep_end'] = pd.to_datetime(df_sleep['ts_end'], format='%d-%m-%Y %H:%M:%S')
        df_sleep['sleep_quality'] = df_sleep['quality'].astype(float)
        df_sleep = df_sleep[['sleep_start', 'sleep_end', 'sleep_quality', 'patient_id']]

    # basal
    df_basal = parse_events(root.find('basal'))
    if not df_basal.empty:
        df_basal['timestamp'] = pd.to_datetime(df_basal['ts'], format='%d-%m-%Y %H:%M:%S')
        df_basal['basal_rate'] = df_basal['value'].astype(float)
        df_basal = df_basal[['timestamp', 'basal_rate', 'patient_id']]

    # temp_basal
    df_temp_basal = parse_events(root.find('temp_basal'))
    if not df_temp_basal.empty:
        df_temp_basal['temp_basal_start'] = pd.to_datetime(df_temp_basal['ts_begin'], format='%d-%m-%Y %H:%M:%S')
        df_temp_basal['temp_basal_end'] = pd.to_datetime(df_temp_basal['ts_end'], format='%d-%m-%Y %H:%M:%S')
        df_temp_basal['temp_basal_rate'] = df_temp_basal['value'].astype(float)
        df_temp_basal = df_temp_basal[['temp_basal_start', 'temp_basal_end', 'temp_basal_rate', 'patient_id']]

    # bolus
    df_bolus = parse_events(root.find('bolus'))
    if not df_bolus.empty:
        df_bolus['timestamp'] = pd.to_datetime(df_bolus['ts_begin'], format='%d-%m-%Y %H:%M:%S')
        df_bolus['bolus_dose'] = df_bolus['dose'].astype(float)
        df_bolus['bwz_carb_input'] = df_bolus.get('bwz_carb_input', pd.Series(0, index=df_bolus.index))
        if 'bwz_carb_input' in df_bolus.columns:
            df_bolus['bwz_carb_input'] = df_bolus['bwz_carb_input'].astype(float)
        df_bolus = df_bolus[['timestamp', 'bolus_dose', 'bwz_carb_input', 'patient_id']]

    return {
        'glucose': df_glucose,
        'meal': df_meal,
        'exercise': df_exercise,
        'heart_rate': df_hr,
        'sleep': df_sleep,
        'basal': df_basal,
        'temp_basal': df_temp_basal,
        'bolus': df_bolus,
        'patient_id': patient_id,
    }

## 2. 特征工程（与训练一致）

In [ ]:
def merge_asof_by_patient(left, right, on, by='patient_id', direction='backward', tolerance=None):
    result_parts = []
    right_cols = [c for c in right.columns if c != by]
    for pid in left[by].unique():
        l_part = left[left[by] == pid].sort_values(on)
        r_part = right[right[by] == pid][right_cols].sort_values(on)
        if r_part.empty:
            result_parts.append(l_part)
            continue
        merged = pd.merge_asof(l_part, r_part, on=on, direction=direction, tolerance=tolerance)
        result_parts.append(merged)
    return pd.concat(result_parts, ignore_index=True)


def build_features(glucose_df, meal_df, exercise_df, sleep_df,
                   bolus_df, basal_df, temp_basal_df, heart_rate_df):
    """构建与训练一致的特征"""
    features_df = glucose_df.copy()
    patient_ids = features_df['patient_id'].unique()

    # IOB
    insulin_peak, insulin_duration = 1.5, 6.0
    iob_values = np.zeros(len(features_df))
    for pid in patient_ids:
        g_mask = features_df['patient_id'] == pid
        g_indices = np.where(g_mask)[0]
        g_times = features_df.loc[g_mask, 'timestamp'].values
        b_data = bolus_df[bolus_df['patient_id'] == pid]
        for _, b_row in b_data.iterrows():
            time_since = (g_times - np.datetime64(b_row['timestamp'])) / np.timedelta64(1, 'h')
            valid = (time_since > 0) & (time_since <= insulin_duration)
            if not np.any(valid):
                continue
            iob_values[g_indices[valid]] += b_row['bolus_dose'] * (
                np.exp(-time_since[valid] / insulin_duration) - np.exp(-time_since[valid] / insulin_peak))
    features_df['IOB'] = np.maximum(0, iob_values)

    # Basal rate
    basal_sub = basal_df[['patient_id', 'timestamp', 'basal_rate']].sort_values(['patient_id', 'timestamp'])
    features_df = merge_asof_by_patient(features_df, basal_sub, on='timestamp')
    features_df['basal_rate'] = features_df['basal_rate'].fillna(0)

    # Temp basal
    features_df['temp_basal_rate'] = 0.0
    for _, tb in temp_basal_df.iterrows():
        mask = ((features_df['patient_id'] == tb['patient_id']) &
                (features_df['timestamp'] >= tb['temp_basal_start']) &
                (features_df['timestamp'] <= tb['temp_basal_end']))
        features_df.loc[mask, 'temp_basal_rate'] = tb['temp_basal_rate']
    features_df['effective_basal_rate'] = np.where(
        features_df['temp_basal_rate'] > 0, features_df['temp_basal_rate'], features_df['basal_rate'])

    # Recent bolus
    features_df['recent_bolus_dose'] = 0.0
    for _, b in bolus_df.iterrows():
        mask_1h = ((features_df['patient_id'] == b['patient_id']) &
                   (features_df['timestamp'] >= b['timestamp']) &
                   (features_df['timestamp'] <= b['timestamp'] + pd.Timedelta(hours=1)))
        features_df.loc[mask_1h, 'recent_bolus_dose'] += b['bolus_dose']

    # COB
    carb_peak, carb_duration = 1.0, 4.0
    cob_values = np.zeros(len(features_df))
    for pid in patient_ids:
        g_mask = features_df['patient_id'] == pid
        g_indices = np.where(g_mask)[0]
        g_times = features_df.loc[g_mask, 'timestamp'].values
        m_data = meal_df[meal_df['patient_id'] == pid]
        for _, m in m_data.iterrows():
            time_since = (g_times - np.datetime64(m['timestamp'])) / np.timedelta64(1, 'h')
            valid = (time_since > 0) & (time_since <= carb_duration)
            if not np.any(valid):
                continue
            cob_values[g_indices[valid]] += m['carbs'] * (
                np.exp(-time_since[valid] / carb_peak) - np.exp(-time_since[valid] / carb_duration))
    features_df['COB'] = np.maximum(0, cob_values)

    # Heart rate
    hr_sub = heart_rate_df[['patient_id', 'timestamp', 'heart_rate']].sort_values(['patient_id', 'timestamp'])
    features_df = merge_asof_by_patient(features_df, hr_sub, on='timestamp', tolerance=pd.Timedelta('30min'))
    features_df['heart_rate'] = features_df['heart_rate'].fillna(70)

    # Exercise intensity
    features_df['exercise_intensity'] = 0.0
    for _, ex in exercise_df.iterrows():
        mask = ((features_df['patient_id'] == ex['patient_id']) &
                (features_df['timestamp'] >= ex['timestamp'] - pd.Timedelta(hours=1)) &
                (features_df['timestamp'] <= ex['timestamp'] + pd.Timedelta(hours=1)))
        intensity = ex['intensity'] * min(ex['duration'] / 60, 1.0) * 2
        features_df.loc[mask, 'exercise_intensity'] = np.maximum(
            features_df.loc[mask, 'exercise_intensity'], intensity)
    features_df['exercise_intensity'] = features_df['exercise_intensity'].clip(0, 10)

    # Sleep quality
    features_df['sleep_quality'] = 3.0
    for _, sl in sleep_df.iterrows():
        mask = ((features_df['patient_id'] == sl['patient_id']) &
                (features_df['timestamp'] >= sl['sleep_start']) &
                (features_df['timestamp'] <= sl['sleep_end'] + pd.Timedelta(hours=12)))
        features_df.loc[mask, 'sleep_quality'] = sl['sleep_quality']

    # ISF
    features_df['hour'] = features_df['timestamp'].dt.hour
    circadian = 1.0 + 0.2 * np.sin(2 * np.pi * (features_df['hour'] - 6) / 24)
    sleep_debt = 5 - features_df['sleep_quality']
    features_df['ISF'] = (1 + sleep_debt * 0.1) * (1 - features_df['exercise_intensity'] * 0.05) * circadian

    # Context features
    features_df['hour_sin'] = np.sin(2 * np.pi * features_df['hour'] / 24)
    features_df['hour_cos'] = np.cos(2 * np.pi * features_df['hour'] / 24)
    features_df['is_dawn'] = features_df['hour'].between(4, 6).astype(int)
    features_df['glucose_zone'] = pd.cut(features_df['glucose_level'],
                                         bins=[0, 70, 140, 180, float('inf')],
                                         labels=[0, 1, 2, 3]).cat.codes

    # delta_glucose for dropna
    features_df['prev_glucose'] = features_df.groupby('patient_id')['glucose_level'].shift(1)
    features_df['delta_glucose'] = features_df['glucose_level'] - features_df['prev_glucose']
    features_df = features_df.dropna(subset=['delta_glucose'])

    return features_df

## 3. 加载模型并选择患者数据

In [ ]:
# 加载模型
checkpoint = torch.load('checkpoints/best_model.pt', map_location=device, weights_only=False)
state_dict = checkpoint['model_state_dict']
hidden_dim = state_dict['context_encoder.0.weight'].shape[0]
context_dim = state_dict['context_encoder.0.weight'].shape[1]
control_dim = state_dict['ode_func.dynamics_net.0.weight'].shape[1] - 1 - hidden_dim

model = NeuralODEGlucosePredictor(
    context_dim=context_dim, hidden_dim=hidden_dim, control_dim=control_dim
).to(device)
model.load_state_dict(state_dict, strict=False)
model.eval()
print(f'模型加载完成: hidden={hidden_dim}, context={context_dim}, control={control_dim}')

In [ ]:
# 选择患者和数据路径
PATIENT_ID = '559'
xml_path = f'OhioT1DM/OhioT1DM 2018/test/{PATIENT_ID}-ws-testing.xml'

data = parse_xml(xml_path)
print(f"患者: {data['patient_id']}")
print(f"  血糖点数: {len(data['glucose'])}")
print(f"  餐食: {len(data['meal'])}, Bolus: {len(data['bolus'])}, 运动: {len(data['exercise'])}")

# 构建特征
features_df = build_features(
    data['glucose'], data['meal'], data['exercise'], data['sleep'],
    data['bolus'], data['basal'], data['temp_basal'], data['heart_rate']
)
features_df = features_df.sort_values('timestamp').reset_index(drop=True)
print(f'特征行数: {len(features_df)}')
print(f'时间范围: {features_df["timestamp"].min()} ~ {features_df["timestamp"].max()}')

## 4. 选择预测窗口并运行预测

选择包含丰富事件（进食、注射、运动）的时间段，用模型预测血糖，与真实CGM对比。

In [ ]:
SEQ_LEN = 24  # 24个15分钟 = 6小时

CONTROL_COLS = ['IOB', 'COB', 'exercise_intensity', 'effective_basal_rate',
                'recent_bolus_dose', 'heart_rate', 'ISF']
CONTEXT_COLS = ['hour_sin', 'hour_cos', 'sleep_quality', 'is_dawn', 'glucose_zone']


def predict_window(features_df, start_idx, seq_len=SEQ_LEN):
    """从 start_idx 开始预测 seq_len 个时间步的血糖"""
    if start_idx + seq_len + 1 > len(features_df):
        return None, None, None, None

    current_row = features_df.iloc[start_idx]
    initial_glucose = torch.tensor([[current_row['glucose_level']]],
                                   dtype=torch.float32, device=device)
    context = torch.tensor(
        [current_row[c] for c in CONTEXT_COLS],
        dtype=torch.float32, device=device).unsqueeze(0)

    # 用未来时间步的真实控制输入（IOB/COB/运动/胰岛素等）
    control_seq = []
    true_glucose = []
    timestamps = []
    for i in range(seq_len):
        future_idx = start_idx + i + 1
        if future_idx >= len(features_df):
            break
        row = features_df.iloc[future_idx]
        control_seq.append([row[c] for c in CONTROL_COLS])
        true_glucose.append(row['glucose_level'])
        timestamps.append(row['timestamp'])

    if len(control_seq) < seq_len:
        return None, None, None, None

    control_seq_t = torch.tensor(control_seq, dtype=torch.float32, device=device).unsqueeze(0)

    with torch.no_grad():
        pred = model(initial_glucose, control_seq_t, context)

    return pred[0].cpu().numpy(), np.array(true_glucose), timestamps, current_row['glucose_level']

In [ ]:
# 自动选择包含丰富事件的窗口
def find_event_rich_windows(features_df, meal_df, bolus_df, exercise_df, seq_len=24, n_windows=3):
    """找到包含最多事件的窗口起始索引"""
    window_scores = []
    for start_idx in range(0, len(features_df) - seq_len - 1, 6):  # 每1.5h步进
        window = features_df.iloc[start_idx:start_idx+seq_len+1]
        t_start = window['timestamp'].iloc[0]
        t_end = window['timestamp'].iloc[-1]

        # 统计窗口内的事件数
        n_meals = len(meal_df[(meal_df['timestamp'] >= t_start) & (meal_df['timestamp'] <= t_end)])
        n_bolus = len(bolus_df[(bolus_df['timestamp'] >= t_start) & (bolus_df['timestamp'] <= t_end)])
        n_exercise = len(exercise_df[(exercise_df['timestamp'] >= t_start) & (exercise_df['timestamp'] <= t_end)])

        score = n_meals + n_bolus + n_exercise
        window_scores.append((start_idx, score, n_meals, n_bolus, n_exercise))

    # 按事件数排序，取前n_windows个（间隔至少seq_len）
    window_scores.sort(key=lambda x: x[1], reverse=True)
    selected = []
    for idx, score, nm, nb, ne in window_scores:
        if len(selected) >= n_windows:
            break
        # 确保不重叠
        if all(abs(idx - s) > seq_len for s in selected):
            selected.append(idx)
    return selected

selected_windows = find_event_rich_windows(
    features_df, data['meal'], data['bolus'], data['exercise'],
    seq_len=SEQ_LEN, n_windows=3
)
print(f'选中 {len(selected_windows)} 个事件丰富窗口: {selected_windows}')

# 对每个窗口运行预测
results = []
for start_idx in selected_windows:
    pred, true, ts, g0 = predict_window(features_df, start_idx)
    if pred is not None:
        results.append({
            'start_idx': start_idx,
            'prediction': pred,
            'true_glucose': true,
            'timestamps': ts,
            'initial_glucose': g0,
        })
        # 计算窗口内误差
        rmse = np.sqrt(np.mean((pred - true) ** 2))
        mae = np.mean(np.abs(pred - true))
        print(f'  窗口 idx={start_idx}: RMSE={rmse:.1f}, MAE={mae:.1f}')
    else:
        print(f'  窗口 idx={start_idx}: 数据不足，跳过')

## 5. 可视化：预测 vs 真实 + 事件标注

In [ ]:
def plot_prediction_vs_real(result, features_df, meal_df, bolus_df, exercise_df, patient_id):
    """绘制预测 vs 真实血糖曲线，标注饮食、胰岛素、运动事件"""
    ts = result['timestamps']
    pred = result['prediction']
    true_g = result['true_glucose']
    g0 = result['initial_glucose']
    start_idx = result['start_idx']

    t_start = ts[0]
    t_end = ts[-1]

    fig, ax = plt.subplots(figsize=(14, 6))

    # 真实血糖
    ax.plot(ts, true_g, 'o-', color='#2196F3', linewidth=2, markersize=4, label='真实血糖 (CGM)')
    # 预测血糖
    ax.plot(ts, pred, 's-', color='#FF5722', linewidth=2, markersize=4, label='模型预测')
    # 初始血糖
    ax.axhline(y=g0, color='gray', linestyle=':', alpha=0.4)

    # 目标范围
    ax.axhspan(70, 180, alpha=0.08, color='green')
    ax.axhline(y=70, color='red', linestyle='--', alpha=0.4, label='低血糖阈值 (70)')
    ax.axhline(y=180, color='orange', linestyle='--', alpha=0.4, label='高血糖阈值 (180)')

    # 标注饮食事件
    meals_in_window = meal_df[(meal_df['timestamp'] >= t_start - pd.Timedelta(minutes=15)) &
                              (meal_df['timestamp'] <= t_end + pd.Timedelta(minutes=15))]
    for _, m in meals_in_window.iterrows():
        ax.axvline(x=m['timestamp'], color='#4CAF50', linestyle='-', alpha=0.6, linewidth=1.5)
        ax.annotate(f"饮食 {m['carbs']:.0f}g",
                    xy=(m['timestamp'], ax.get_ylim()[1] * 0.95),
                    fontsize=9, color='#4CAF50', fontweight='bold',
                    ha='center', va='top',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='#4CAF50', alpha=0.8))

    # 标注胰岛素事件
    bolus_in_window = bolus_df[(bolus_df['timestamp'] >= t_start - pd.Timedelta(minutes=15)) &
                               (bolus_df['timestamp'] <= t_end + pd.Timedelta(minutes=15))]
    for _, b in bolus_in_window.iterrows():
        ax.axvline(x=b['timestamp'], color='#9C27B0', linestyle='-', alpha=0.6, linewidth=1.5)
        ax.annotate(f"胰岛素 {b['bolus_dose']:.1f}U",
                    xy=(b['timestamp'], ax.get_ylim()[1] * 0.85),
                    fontsize=9, color='#9C27B0', fontweight='bold',
                    ha='center', va='top',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='#9C27B0', alpha=0.8))

    # 标注运动事件
    ex_in_window = exercise_df[(exercise_df['timestamp'] >= t_start - pd.Timedelta(hours=1)) &
                               (exercise_df['timestamp'] <= t_end + pd.Timedelta(hours=1))]
    for _, e in ex_in_window.iterrows():
        ax.axvline(x=e['timestamp'], color='#FF9800', linestyle='-', alpha=0.6, linewidth=1.5)
        ax.annotate(f"运动 {e['duration']:.0f}min",
                    xy=(e['timestamp'], ax.get_ylim()[1] * 0.75),
                    fontsize=9, color='#FF9800', fontweight='bold',
                    ha='center', va='top',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='#FF9800', alpha=0.8))

    # 误差统计
    rmse = np.sqrt(np.mean((pred - true_g) ** 2))
    mae = np.mean(np.abs(pred - true_g))
    mard = np.mean(np.abs(pred - true_g) / np.maximum(true_g, 1)) * 100

    ax.set_xlabel('时间', fontsize=12)
    ax.set_ylabel('血糖 (mg/dL)', fontsize=12)
    ts0 = pd.Timestamp(ts[0])
    ax.set_title(f'患者 {patient_id}: 预测 vs 真实  |  RMSE={rmse:.1f}, MAE={mae:.1f}, MARD={mard:.1f}%\n'
                 f'起始: {ts0.strftime("%m/%d %H:%M")} (G0={g0:.0f} mg/dL)', fontsize=13)
    ax.legend(loc='upper right', fontsize=10)
    ax.set_ylim(40, 300)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 误差随时间变化
    fig2, ax2 = plt.subplots(figsize=(14, 3))
    time_min = np.arange(len(pred)) * 15
    error = pred - true_g
    ax2.bar(time_min, error, width=12, color=['#4CAF50' if e >= 0 else '#FF5722' for e in error], alpha=0.7)
    ax2.axhline(y=0, color='black', linewidth=0.8)
    ax2.set_xlabel('预测时间 (分钟)', fontsize=12)
    ax2.set_ylabel('预测误差 (mg/dL)', fontsize=12)
    ax2.set_title(f'逐时间步预测误差 (正值=高估, 负值=低估)', fontsize=13)
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# 绘制所有选中窗口
for result in results:
    plot_prediction_vs_real(result, features_df, data['meal'], data['bolus'], data['exercise'], data['patient_id'])

## 6. 交互式预测：自定义输入

你可以指定：当前血糖、进食碳水、注射胰岛素剂量、运动强度，来预测未来血糖变化。

控制向量: [IOB, COB, exercise_intensity, effective_basal_rate, recent_bolus_dose, heart_rate, ISF]

In [ ]:
def interactive_predict(
    initial_glucose=120,
    hour=8,
    sleep_quality=3,
    # 饮食输入
    carbs=60,            # 碳水克数
    # 胰岛素输入
    bolus_dose=4.0,      # bolus剂量 (U)
    basal_rate=0.8,      # 基础率 (U/h)
    # 运动输入
    exercise_intensity=0, # 运动强度 (0-10)
    # 其他
    heart_rate=72,
    isf=1.0,
    seq_len=24,          # 预测6小时
):
    """
    交互式血糖预测
    
    参数:
        initial_glucose: 当前血糖 (mg/dL)
        hour: 当前时间 (小时, 0-23)
        sleep_quality: 睡眠质量 (1-5)
        carbs: 碳水摄入量 (g)
        bolus_dose: 大剂量胰岛素 (U)
        basal_rate: 基础胰岛素率 (U/h)
        exercise_intensity: 运动强度 (0-10, 0=无运动)
        heart_rate: 心率
        isf: 胰岛素敏感因子
        seq_len: 预测步数 (每步15分钟)
    """
    # 上下文
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)
    is_dawn = 1 if 4 <= hour <= 6 else 0
    glucose_zone = 0 if initial_glucose < 70 else (1 if initial_glucose < 140 else (2 if initial_glucose < 180 else 3))
    context = torch.tensor([[hour_sin, hour_cos, sleep_quality, is_dawn, glucose_zone]],
                          dtype=torch.float32, device=device)

    # IOB衰减曲线 (bolus后随时间衰减)
    iob_list = [bolus_dose * (np.exp(-i * 0.25 / 2.0)) for i in range(seq_len)]
    # COB吸收曲线 (碳水双指数吸收)
    cob_list = [carbs * (0.7 * np.exp(-i * 0.25 / 0.5) + 0.3 * np.exp(-i * 0.25 / 3.0)) for i in range(seq_len)]
    # 运动强度（假设运动持续1小时后衰减）
    if exercise_intensity > 0:
        ex_list = [exercise_intensity if i < 4 else exercise_intensity * np.exp(-(i-4)*0.5) for i in range(seq_len)]
    else:
        ex_list = [0.0] * seq_len

    # 构建控制序列
    control = torch.zeros(1, seq_len, 7, dtype=torch.float32, device=device)
    control[0, :, 0] = torch.tensor(iob_list)
    control[0, :, 1] = torch.tensor(cob_list)
    control[0, :, 2] = torch.tensor(ex_list)
    control[0, :, 3] = basal_rate
    control[0, :, 4] = bolus_dose
    control[0, :, 5] = heart_rate
    control[0, :, 6] = isf

    # 预测
    g0 = torch.tensor([[initial_glucose]], dtype=torch.float32, device=device)
    with torch.no_grad():
        pred = model(g0, control, context)
    pred_np = pred[0].cpu().numpy()

    # 绘图
    time_min = np.arange(seq_len) * 15
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(time_min, pred_np, 'o-', color='#FF5722', linewidth=2.5, markersize=5, label='预测血糖')
    ax.axhline(y=initial_glucose, color='gray', linestyle=':', alpha=0.5, label=f'初始血糖 ({initial_glucose:.0f})')
    ax.axhspan(70, 180, alpha=0.08, color='green', label='目标范围 (70-180)')
    ax.axhline(y=70, color='red', linestyle='--', alpha=0.4, label='低血糖阈值')
    ax.axhline(y=180, color='orange', linestyle='--', alpha=0.4, label='高血糖阈值')

    # 标注输入
    input_text = f'输入: 碳水={carbs}g, 胰岛素={bolus_dose}U, 基础率={basal_rate}U/h, 运动={exercise_intensity}'
    ax.text(0.02, 0.98, input_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    peak_idx = np.argmax(np.abs(pred_np - initial_glucose))
    peak_val = pred_np[peak_idx]
    peak_time = time_min[peak_idx]
    direction = '↑' if peak_val > initial_glucose else '↓'
    ax.annotate(f'{direction} {peak_val:.0f} mg/dL\n({peak_time}min后)',
                xy=(peak_time, peak_val), xytext=(peak_time + 30, peak_val + 15),
                fontsize=10, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='red'),
                bbox=dict(boxstyle='round', facecolor='lightyellow'))

    ax.set_xlabel('时间 (分钟)', fontsize=12)
    ax.set_ylabel('血糖 (mg/dL)', fontsize=12)
    ax.set_title(f'交互式血糖预测 (起始 {hour}:00, 血糖 {initial_glucose} mg/dL)', fontsize=13)
    ax.legend(loc='upper right', fontsize=9)
    ax.set_ylim(40, 300)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f'\n预测结果:')
    print(f'  起始血糖:   {initial_glucose:.0f} mg/dL')
    print(f'  30分钟后:   {pred_np[1]:.1f} mg/dL')
    print(f'  1小时后:    {pred_np[3]:.1f} mg/dL')
    print(f'  2小时后:    {pred_np[7]:.1f} mg/dL')
    print(f'  4小时后:    {pred_np[15]:.1f} mg/dL')
    print(f'  6小时后:    {pred_np[-1]:.1f} mg/dL')
    if pred_np.max() > 180:
        print(f'  ⚠ 高血糖风险: 峰值 {pred_np.max():.1f} mg/dL')
    if pred_np.min() < 70:
        print(f'  ⚠ 低血糖风险: 谷值 {pred_np.min():.1f} mg/dL')

    return pred_np

print('交互式预测函数已定义。调用方式:')
print('interactive_predict(initial_glucose=120, carbs=60, bolus_dose=4, exercise_intensity=0)')

### 场景演示

In [ ]:
# 场景1：早餐后 — 60g碳水 + 4U胰岛素
print('=== 场景1：早餐后 (60g碳水 + 4U胰岛素) ===')
pred1 = interactive_predict(
    initial_glucose=110,
    hour=8,
    carbs=60,
    bolus_dose=4.0,
    basal_rate=0.8,
    exercise_intensity=0,
)

In [ ]:
# 场景2：运动日 — 60g碳水 + 4U胰岛素 + 中等强度运动
print('=== 场景2：运动日 (60g碳水 + 4U胰岛素 + 运动) ===')
pred2 = interactive_predict(
    initial_glucose=110,
    hour=8,
    carbs=60,
    bolus_dose=4.0,
    basal_rate=0.8,
    exercise_intensity=5,
)

In [ ]:
# 场景3：高碳水少胰岛素 — 血糖飙升风险
print('=== 场景3：高碳水少胰岛素 (100g碳水 + 2U胰岛素) ===')
pred3 = interactive_predict(
    initial_glucose=150,
    hour=12,
    carbs=100,
    bolus_dose=2.0,
    basal_rate=0.8,
    exercise_intensity=0,
)

## 7. 真实数据验证：自动选择有完整事件的日期

从测试数据中选择有完整事件链的真实时间段，展示模型在真实场景下的预测能力。

In [ ]:
# 选择一个完整日期进行全天预测
def predict_full_day(features_df, meal_df, bolus_df, exercise_df, patient_id,
                      date_str=None, seq_len=8, step=2):
    """
    对一个完整日期进行滚动预测
    seq_len: 每次预测2小时(8步)
    step: 每30分钟(2步)做一次新预测
    """
    if date_str:
        day_mask = features_df['timestamp'].dt.strftime('%Y-%m-%d') == date_str
        day_df = features_df[day_mask].reset_index(drop=True)
    else:
        # 选择数据最多的日期
        day_counts = features_df.groupby(features_df['timestamp'].dt.date).size()
        best_date = day_counts.idxmax()
        date_str = str(best_date)
        day_df = features_df[features_df['timestamp'].dt.date == best_date].reset_index(drop=True)

    if len(day_df) < seq_len + 1:
        print(f'日期 {date_str} 数据不足 ({len(day_df)} 行)')
        return

    print(f'预测日期: {date_str}, 数据点: {len(day_df)}')

    # 滚动预测
    all_pred_times = []
    all_pred_vals = []
    all_true_times = list(day_df['timestamp'])
    all_true_vals = list(day_df['glucose_level'])

    with torch.no_grad():
        for start in range(0, len(day_df) - seq_len, step):
            row = day_df.iloc[start]
            g0 = torch.tensor([[row['glucose_level']]], dtype=torch.float32, device=device)
            ctx = torch.tensor(
                [row[c] for c in CONTEXT_COLS],
                dtype=torch.float32, device=device).unsqueeze(0)

            ctrl_seq = []
            for j in range(seq_len):
                fj = start + j + 1
                if fj >= len(day_df):
                    break
                ctrl_seq.append([day_df.iloc[fj][c] for c in CONTROL_COLS])

            if len(ctrl_seq) < seq_len:
                continue

            ctrl_t = torch.tensor(ctrl_seq, dtype=torch.float32, device=device).unsqueeze(0)
            pred = model(g0, ctrl_t, ctx)[0].cpu().numpy()

            # 记录预测起始点对应2小时后的值
            for j in range(seq_len):
                fj = start + j + 1
                if fj < len(day_df):
                    all_pred_times.append(day_df.iloc[fj]['timestamp'])
                    all_pred_vals.append(pred[j])

    # 绘图
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.plot(all_true_times, all_true_vals, '-', color='#2196F3', linewidth=2, label='真实血糖 (CGM)')
    ax.plot(all_pred_times, all_pred_vals, '.', color='#FF5722', markersize=3, alpha=0.5, label='模型预测')

    ax.axhspan(70, 180, alpha=0.08, color='green')
    ax.axhline(y=70, color='red', linestyle='--', alpha=0.4)
    ax.axhline(y=180, color='orange', linestyle='--', alpha=0.4)

    # 标注事件
    t_start = day_df['timestamp'].iloc[0]
    t_end = day_df['timestamp'].iloc[-1]
    for _, m in meal_df[(meal_df['timestamp'] >= t_start) & (meal_df['timestamp'] <= t_end)].iterrows():
        ax.axvline(x=m['timestamp'], color='#4CAF50', linestyle='-', alpha=0.5, linewidth=1.5)
        ax.annotate(f"{m['carbs']:.0f}g", xy=(m['timestamp'], 280),
                    fontsize=8, color='#4CAF50', ha='center', fontweight='bold')
    for _, b in bolus_df[(bolus_df['timestamp'] >= t_start) & (bolus_df['timestamp'] <= t_end)].iterrows():
        ax.axvline(x=b['timestamp'], color='#9C27B0', linestyle='-', alpha=0.5, linewidth=1.5)
        ax.annotate(f"{b['bolus_dose']:.1f}U", xy=(b['timestamp'], 260),
                    fontsize=8, color='#9C27B0', ha='center', fontweight='bold')
    for _, e in exercise_df[(exercise_df['timestamp'] >= t_start) & (exercise_df['timestamp'] <= t_end)].iterrows():
        ax.axvline(x=e['timestamp'], color='#FF9800', linestyle='-', alpha=0.5, linewidth=1.5)
        ax.annotate(f"运动{e['duration']:.0f}min", xy=(e['timestamp'], 240),
                    fontsize=8, color='#FF9800', ha='center', fontweight='bold')

    # 添加事件图例说明
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='#2196F3', linewidth=2, label='真实血糖'),
        Line2D([0], [0], marker='.', color='#FF5722', linewidth=0, markersize=8, label='模型预测'),
        Line2D([0], [0], color='#4CAF50', linewidth=1.5, label='饮食'),
        Line2D([0], [0], color='#9C27B0', linewidth=1.5, label='胰岛素'),
        Line2D([0], [0], color='#FF9800', linewidth=1.5, label='运动'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9)

    ax.set_xlabel('时间', fontsize=12)
    ax.set_ylabel('血糖 (mg/dL)', fontsize=12)
    ax.set_title(f'患者 {patient_id} | {date_str} 全天预测 vs 真实', fontsize=13)
    ax.set_ylim(40, 300)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 计算全天误差
    pred_arr = np.array(all_pred_vals)
    # 对应的真实值
    true_arr = []
    for pt in all_pred_times:
        idx = day_df[day_df['timestamp'] == pt].index
        if len(idx) > 0:
            true_arr.append(day_df.loc[idx[0], 'glucose_level'])
    true_arr = np.array(true_arr)
    if len(pred_arr) == len(true_arr):
        rmse = np.sqrt(np.mean((pred_arr - true_arr) ** 2))
        mae = np.mean(np.abs(pred_arr - true_arr))
        mard = np.mean(np.abs(pred_arr - true_arr) / np.maximum(true_arr, 1)) * 100
        print(f'全天误差: RMSE={rmse:.1f}, MAE={mae:.1f}, MARD={mard:.1f}%')

# 运行全天预测
predict_full_day(features_df, data['meal'], data['bolus'], data['exercise'], data['patient_id'])

## 8. 自定义真实数据预测

选择特定时间段，查看模型在真实事件序列下的预测 vs 真实血糖。

In [ ]:
# 修改日期字符串来选择你想要的日期
CUSTOM_DATE = None  # 设为如 '2018-03-15' 来选择特定日期，None=自动选择数据最全的日期

# 也可以切换患者
OTHER_PATIENT = None  # 设为如 '563' 来切换患者

if OTHER_PATIENT:
    xml_path2 = f'OhioT1DM/OhioT1DM 2018/test/{OTHER_PATIENT}-ws-testing.xml'
    data2 = parse_xml(xml_path2)
    features2 = build_features(
        data2['glucose'], data2['meal'], data2['exercise'], data2['sleep'],
        data2['bolus'], data2['basal'], data2['temp_basal'], data2['heart_rate']
    )
    features2 = features2.sort_values('timestamp').reset_index(drop=True)
    predict_full_day(features2, data2['meal'], data2['bolus'], data2['exercise'], data2['patient_id'],
                     date_str=CUSTOM_DATE)
else:
    predict_full_day(features_df, data['meal'], data['bolus'], data['exercise'], data['patient_id'],
                     date_str=CUSTOM_DATE)